In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/network-intrusion-dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Tuesday-WorkingHours.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Monday-WorkingHours.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
/kaggle/input/network-intrusion-dataset/Wednesday-workingHours.pcap_ISCX.csv
/kaggle/input/cicddos2019/Syn-training.parquet
/kaggle/input/cicddos2019/UDPLag-testing.parquet
/kaggle/input/cicddos2019/NetBIOS-testing.parquet
/kaggle/input/cicddos2019/Portmap-training.parquet
/kaggle/input/cicddos2019/Syn-testing.parquet
/kaggle/input/cicddos2019/MSSQL-testing.parquet
/kaggle/input/cicddos201

# Start


In [ ]:
# Cell 1: imports & config
import os, glob, json
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
import joblib
import warnings
warnings.filterwarnings("ignore")

# Paths (adjust if needed)
CIC_DIR = "/kaggle/input/cicddos2019"
UNSW_DIR = "/kaggle/input/unsw-nb15"
NET_INTR_DIR = "/kaggle/input/network-intrusion-dataset"
OUT_DIR = "/kaggle/working/dataset_explore"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


In [ ]:
# Cell 2: robust CSV reader helper
import chardet
def read_csv_robust(path, nbytes=10000, **kwargs):
    # try to detect encoding first
    with open(path, "rb") as f:
        raw = f.read(nbytes)
    try:
        enc = chardet.detect(raw)['encoding'] or 'utf-8'
    except Exception:
        enc = 'utf-8'
    for encoding in [enc, 'utf-8', 'latin1', 'iso-8859-1', 'cp1252']:
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False, **kwargs)
        except Exception as e:
            last_exc = e
    # last resort: read with errors replaced
    return pd.read_csv(path, encoding='utf-8', error_bad_lines=False, engine='python', low_memory=False, **kwargs)


In [ ]:
# Cell 3: quick explore CIC parquet + UNSW CSVs + network csvs
out_summary = {}

# 1) CICDDoS2019 (parquet files)
cic_files = sorted(glob.glob(os.path.join(CIC_DIR, "*.parquet")))
print("Found CIC parquet files:", len(cic_files))
# load iteratively and concat
cic_parts = []
for f in cic_files:
    print("Reading:", os.path.basename(f))
    cic_parts.append(pd.read_parquet(f))
cic = pd.concat(cic_parts, ignore_index=True)
print("CIC shape:", cic.shape)




Found CIC parquet files: 17
Reading: DNS-testing.parquet
Reading: LDAP-testing.parquet
Reading: LDAP-training.parquet
Reading: MSSQL-testing.parquet
Reading: MSSQL-training.parquet
Reading: NTP-testing.parquet
Reading: NetBIOS-testing.parquet
Reading: NetBIOS-training.parquet
Reading: Portmap-training.parquet
Reading: SNMP-testing.parquet
Reading: Syn-testing.parquet
Reading: Syn-training.parquet
Reading: TFTP-testing.parquet
Reading: UDP-testing.parquet
Reading: UDP-training.parquet
Reading: UDPLag-testing.parquet
Reading: UDPLag-training.parquet
CIC shape: (431371, 78)


In [ ]:
# 3) network-intrusion-dataset (many CSVs)
net_files = sorted(glob.glob(os.path.join(NET_INTR_DIR, "/kaggle/input/network-intrusion-dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")))
print("Network intrusion csvs:", len(net_files))

net_parts = []

for f in net_files:
    print("Reading:", os.path.basename(f))
    
    df = read_csv_robust(f)      # đọc toàn bộ CSV
    # df = df.sample(frac=0.2, random_state=RANDOM_STATE)  # nếu muốn sample 20%
    
    net_parts.append(df)

net = pd.concat(net_parts, ignore_index=True)
print("Network merged shape:", net.shape)


Network intrusion csvs: 1
Reading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Network merged shape: (225745, 79)


In [ ]:
import pandas as pd
import numpy as np

def explore_dataset(name, df, max_rows=5):
    print("="*120)
    print(f"📌 DATASET: {name}")
    print("="*120)

    # 1) Basic info
    print(f"➡ Shape: {df.shape}")
    print("\n➡ Sample rows:")
    display(df.head(max_rows))

    # 2) Columns
    print("\n➡ Columns:")
    print(df.columns.tolist())

    # 3) Missing values
    missing = df.isna().sum()
    if missing.sum() == 0:
        print("\n➡ Missing values: 0 ✔")
    else:
        print("\n➡ Missing values:")
        display(missing[missing > 0])

    # 4) Infinite values
    inf_count = np.isinf(df.select_dtypes(include=[np.number])).sum()
    if inf_count.sum() == 0:
        print("\n➡ Infinite values: 0 ✔")
    else:
        print("\n➡ Infinite values detected:")
        display(inf_count[inf_count > 0])

    # 5) Duplicates
    dup = df.duplicated().sum()
    print(f"\n➡ Duplicate rows: {dup}")

    # 6) Check Label column
    label_candidates = ["Label", "Attack", "attack_cat", "class", "label"]
    found = None
    for c in label_candidates:
        if c in df.columns:
            found = c
            break

    if found:
        print(f"\n➡ Detected label column: {found}")
        print(df[found].value_counts())
        print("\n➡ Label distribution (%):")
        print((df[found].value_counts(normalize=True) * 100).round(3))
    else:
        print("\n⚠ No label column detected.")

    # 7) Numeric summary
    print("\n➡ Numeric Feature Summary:")
    display(df.describe().T.head(10))  # show 10 dòng đầu

    print("\n✔ Exploration complete.\n\n")



In [ ]:
display(cic.head(5))

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,17,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
1,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
3,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
4,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS


In [ ]:
import pandas as pd
import numpy as np
import re

def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()                # remove leading/trailing spaces
        .str.replace(' ', '_')      # replace spaces with underscore
        .str.replace('[^A-Za-z0-9_]', '', regex=True)  # keep only alphanum + underscore
        .str.lower()                # lowercase
    )
    return df


In [ ]:
cic_normalize = normalize_columns(cic)          # CICDDoS2019
cicids_normalize = normalize_columns(net)    # CICIDS2017
cic_normalize.head(10)

,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,17,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
1,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
3,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
4,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
5,17,1,2,0,2736.0,0.0,1368.0,1368.0,1368.0,0.0,...,1472,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
6,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
7,17,232,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
8,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS
9,17,11,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DrDoS_DNS


In [ ]:
def clean_df(df):
    df = df.copy()

    # Replace inf with nan
    df = df.replace([np.inf, -np.inf], np.nan)

    # Impute numeric missing with median
    num_cols = df.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())

    # Drop duplicates
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {before - len(df)} duplicates")

    return df

cic_cl_duplicates = clean_df(cic_normalize)
cicids_cl_duplicates = clean_df(cicids_normalize)


Removed 5295 duplicates
Removed 2633 duplicates


In [ ]:
def map_label_cic_2019(x):
    x = str(x).lower()
    if x == "benign":
        return 0
    return 1

cic_cl_duplicates['label'] = cic_cl_duplicates['label'].apply(map_label_cic_2019)

cic_cl_duplicates.head(5)

,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,17,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [ ]:
dos_keywords = ['dos', 'ddos', 'hulk', 'goldeneye', 'slowloris', 'slowhttptest']

def map_label_cicids(x):
    x = str(x).lower()
    if x == "benign":
        return 0
    return 1

cicids_cl_duplicates['label'] = cicids_cl_duplicates['label'].apply(map_label_cicids)
cicids_cl_duplicates.head(5)


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0


In [ ]:
if 'cic' in globals():
    explore_dataset("CICDDoS2019", cic_cl_duplicates)

if 'net' in globals():
    explore_dataset("CICIDS2017", cicids_cl_duplicates)


📌 DATASET: CICDDoS2019
➡ Shape: (426076, 78)

➡ Sample rows:


,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,17,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1



➡ Columns:
['protocol', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'fwd_packets_length_total', 'bwd_packets_length_total', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytess', 'flow_packetss', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packetss', 'bwd_packetss', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'do

,count,mean,std,min,25%,50%,75%,max
protocol,426076.0,1.402451e+01,4.925998e+00,0.0,6.00,17.000000,17.0,1.700000e+01
flow_duration,426076.0,8.509244e+06,2.137694e+07,1.0,855.00,46104.500000,3002897.0,1.199987e+08
total_fwd_packets,426076.0,2.441329e+01,1.970868e+02,1.0,4.00,5.000000,16.0,8.666600e+04
total_backward_packets,426076.0,2.492201e+00,5.671900e+01,0.0,0.00,0.000000,2.0,3.170000e+04
fwd_packets_length_total,426076.0,9.532451e+03,3.472593e+04,0.0,84.00,2064.000000,5280.0,1.526642e+07
bwd_packets_length_total,426076.0,1.653023e+03,1.070495e+05,0.0,0.00,0.000000,0.0,5.842950e+07
fwd_packet_length_max,426076.0,3.611496e+02,3.197199e+02,0.0,38.00,440.000000,516.0,3.212000e+04
fwd_packet_length_min,426076.0,2.977724e+02,2.726888e+02,0.0,6.00,330.000000,516.0,2.131000e+03
fwd_packet_length_mean,426076.0,3.282580e+02,2.677223e+02,0.0,32.25,429.714294,516.0,3.015291e+03
fwd_packet_length_std,426076.0,2.034486e+01,7.097531e+01,0.0,0.00,0.000000,0.0,2.221556e+03



✔ Exploration complete.


📌 DATASET: CICIDS2017
➡ Shape: (223112, 79)

➡ Sample rows:


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0



➡ Columns:
['destination_port', 'flow_duration', 'total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytess', 'flow_packetss', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packetss', 'bwd_packetss', 'min_packet_length', 'max_packet_length', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_fl

,count,mean,std,min,25%,50%,75%,max
destination_port,223112.0,8.982618e+03,1.984753e+04,0.0,80.00,8.000000e+01,8.000000e+01,6.553200e+04
flow_duration,223112.0,1.643322e+07,3.166018e+07,-1.0,81637.75,1.536448e+06,8.957830e+06,1.199999e+08
total_fwd_packets,223112.0,4.905375e+00,1.550975e+01,1.0,2.00,3.000000e+00,5.000000e+00,1.932000e+03
total_backward_packets,223112.0,4.611554e+00,2.188016e+01,0.0,1.00,4.000000e+00,5.000000e+00,2.942000e+03
total_length_of_fwd_packets,223112.0,9.496634e+02,3.267081e+03,0.0,26.00,3.000000e+01,6.200000e+01,1.830120e+05
total_length_of_bwd_packets,223112.0,6.029361e+03,3.944391e+04,0.0,0.00,1.640000e+02,1.160100e+04,5.172346e+06
fwd_packet_length_max,223112.0,5.445760e+02,1.874258e+03,0.0,6.00,2.000000e+01,3.500000e+01,1.168000e+04
fwd_packet_length_min,223112.0,2.790173e+01,1.642428e+02,0.0,0.00,0.000000e+00,6.000000e+00,1.472000e+03
fwd_packet_length_mean,223112.0,1.664596e+02,5.076248e+02,0.0,6.00,8.666667e+00,3.200000e+01,3.867000e+03
fwd_packet_length_std,223112.0,2.174395e+02,8.017597e+02,0.0,0.00,5.301991e+00,1.026320e+01,6.692645e+03



✔ Exploration complete.




In [ ]:
cic_clean1 = clean_df(cic_cl_duplicates)
cicids_clean1 = clean_df(cicids_cl_duplicates)
cic_clean1.head()

Removed 7163 duplicates
Removed 0 duplicates


,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,17,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,17,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,17,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,...,1480,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,17,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,...,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [ ]:
column_map = {
    "fwd_packets_length_total": "total_length_of_fwd_packets",
    "bwd_packets_length_total": "total_length_of_bwd_packets",
    "packet_length_min": "min_packet_length",
    "packet_length_max": "max_packet_length",
    "avg_packet_size": "average_packet_size",
    "init_fwd_win_bytes": "init_win_bytes_forward",
    "init_bwd_win_bytes": "init_win_bytes_backward",
    "fwd_act_data_packets": "act_data_pkt_fwd",
    "fwd_seg_size_min": "min_seg_size_forward",
}


In [ ]:
cicids_renamed = cicids_clean1.copy()

reverse_map = {v: k for k, v in column_map.items()}  # map từ cicids → cicddos2019

cicids_renamed = cicids_renamed.rename(columns=reverse_map)


In [ ]:
common_features = list(
    set(cic_clean1.columns) &
    set(cicids_renamed.columns)
)

# bỏ label
common_features.remove('label')

print("Common Features: ", len(common_features))

missing_in_cic = [c for c in common_features if c not in cic_clean1.columns]
missing_in_cicids = [c for c in common_features if c not in cicids_renamed.columns]

print("Missing in CICDDoS2019:", missing_in_cic)
print("Missing in CICIDS2017:", missing_in_cicids)


Common Features:  76
Missing in CICDDoS2019: []
Missing in CICIDS2017: []


In [ ]:
# Make sure common_features does NOT include label
common_features = [c for c in common_features if c != 'label']

# Apply to CICDDoS2019
cic_cleaned = cic_clean1[cic_clean1.columns.intersection(common_features + ['label'])].copy()

# Apply to CICIDS2017
cicids_cleaned = cicids_renamed[cicids_renamed.columns.intersection(common_features + ['label'])].copy()

print("CICDDoS2019 cleaned shape:", cic_cleaned.shape)
print("CICIDS2017 cleaned shape:", cicids_cleaned.shape)


CICDDoS2019 cleaned shape: (418913, 77)
CICIDS2017 cleaned shape: (223112, 77)


In [ ]:
def clean_negative_and_remove_unimportant(df):
    df = df.copy()

    # 1) Drop low-importance columns
    cols_to_drop = [
        "init_fwd_win_bytes",
        "init_bwd_win_bytes",
        "fwd_seg_size_min"
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    # 2) Columns that must never be negative → clip to 0
    columns_never_negative = [
        "flow_duration",
        "flow_bytess",
        "flow_packetss",
        "flow_iat_min", "flow_iat_max", "flow_iat_mean",
        "fwd_iat_min",
        "fwd_header_length", "bwd_header_length",
        "packet_length_min", "packet_length_max",
        "packet_length_mean", "packet_length_std",
        "packet_length_variance",
    ]

    for col in columns_never_negative:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)

    return df


In [ ]:
cic_cleaned_rm = clean_negative_and_remove_unimportant(cic_cleaned)
cicids_cleaned_rm = clean_negative_and_remove_unimportant(cicids_cleaned)

cic_cleaned_rm.shape,cicids_cleaned_rm.shape 

((418913, 74), (223112, 74))

In [ ]:
import os

OUTPUT_DIR = "processed_datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(cic_cleaned_rm.shape)
# CICDDoS2019 final
cic_final = cic_cleaned_rm
cic_final_path = f"{OUTPUT_DIR}/cicddos2019_binary_processed.csv"
cic_final.to_csv(cic_final_path, index=False)

print(cicids_cleaned_rm.shape)
# CICIDS2017 final
cicids_final = cicids_cleaned_rm
cicids_final_path = f"{OUTPUT_DIR}/cicids2017_binary_processed.csv"
cicids_final.to_csv(cicids_final_path, index=False)

print("Saved processed datasets:")
print(" -", cic_final_path, cic_final.shape)
print(" -", cicids_final_path, cicids_final.shape)


(418913, 74)
(223112, 74)
Saved processed datasets:
 - processed_datasets/cicddos2019_binary_processed.csv (418913, 74)
 - processed_datasets/cicids2017_binary_processed.csv (223112, 74)


In [ ]:
import pandas as pd

cic_path = "processed_datasets/cicddos2019_binary_processed.csv"
cicids_path = "processed_datasets/cicids2017_binary_processed.csv"

# Load dataset
cic_proc = pd.read_csv(cic_path)
cicids_proc = pd.read_csv(cicids_path)

print("Loaded processed datasets:")
print(" - CICDDoS2019:", cic_proc.shape)
print(" - CICIDS2017 :", cicids_proc.shape)


display(cic_proc.head())
display(cic_proc.head())

Loaded processed datasets:
 - CICDDoS2019: (418913, 74)
 - CICIDS2017 : (223112, 74)


,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,...,fwd_act_data_packets,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,...,fwd_act_data_packets,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [ ]:
def check_negative_values(df, name="dataset"):
    neg_cols = {}
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            neg_cols[col] = neg_count

    if len(neg_cols) == 0:
        print(f"✓ {name}: Không có giá trị âm.")
    else:
        print(f"⚠ {name}: Có giá trị âm ở các cột sau:")
        for col, count in neg_cols.items():
            print(f" - {col}: {count} values")

# Check both processed datasets
check_negative_values(cic_proc, "CICDDoS2019")
check_negative_values(cicids_proc, "CICIDS2017")


✓ CICDDoS2019: Không có giá trị âm.
✓ CICIDS2017: Không có giá trị âm.


In [ ]:

explore_dataset("CICDDoS2019", cic_proc)
explore_dataset("CICIDS2017", cicids_proc)


📌 DATASET: CICDDoS2019
➡ Shape: (418913, 74)

➡ Sample rows:


,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,...,fwd_act_data_packets,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1



➡ Columns:
['flow_duration', 'total_fwd_packets', 'total_backward_packets', 'fwd_packets_length_total', 'bwd_packets_length_total', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytess', 'flow_packetss', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packetss', 'bwd_packetss', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'downup_ratio',

,count,mean,std,min,25%,50%,75%,max
flow_duration,418913.0,8.654648e+06,2.152974e+07,1.0,961.0,54257.00000,3003598.0,1.199987e+08
total_fwd_packets,418913.0,2.479473e+01,1.987429e+02,1.0,4.0,6.00000,18.0,8.666600e+04
total_backward_packets,418913.0,2.533722e+00,5.720095e+01,0.0,0.0,0.00000,2.0,3.170000e+04
fwd_packets_length_total,418913.0,9.669774e+03,3.492898e+04,0.0,82.0,2064.00000,5440.0,1.526642e+07
bwd_packets_length_total,418913.0,1.681288e+03,1.079759e+05,0.0,0.0,0.00000,0.0,5.842950e+07
fwd_packet_length_max,418913.0,3.547905e+02,3.126643e+02,0.0,38.0,440.00000,516.0,3.212000e+04
fwd_packet_length_min,418913.0,2.903754e+02,2.621954e+02,0.0,6.0,330.00000,516.0,2.131000e+03
fwd_packet_length_mean,418913.0,3.213595e+02,2.576453e+02,0.0,32.0,428.92307,516.0,3.015291e+03
fwd_packet_length_std,418913.0,2.066632e+01,7.164219e+01,0.0,0.0,0.00000,0.0,2.221556e+03
bwd_packet_length_max,418913.0,8.186213e+01,4.977494e+02,0.0,0.0,0.00000,0.0,3.796000e+04



✔ Exploration complete.


📌 DATASET: CICIDS2017
➡ Shape: (223112, 74)

➡ Sample rows:


,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,...,fwd_act_data_packets,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,3,2,0,12,0,6,6,6.0,0.0,0,...,1,0.0,0.0,0,0,0.0,0.0,0,0,0
1,109,1,1,6,6,6,6,6.0,0.0,6,...,0,0.0,0.0,0,0,0.0,0.0,0,0,0
2,52,1,1,6,6,6,6,6.0,0.0,6,...,0,0.0,0.0,0,0,0.0,0.0,0,0,0
3,34,1,1,6,6,6,6,6.0,0.0,6,...,0,0.0,0.0,0,0,0.0,0.0,0,0,0
4,3,2,0,12,0,6,6,6.0,0.0,0,...,1,0.0,0.0,0,0,0.0,0.0,0,0,0



➡ Columns:
['flow_duration', 'total_fwd_packets', 'total_backward_packets', 'fwd_packets_length_total', 'bwd_packets_length_total', 'fwd_packet_length_max', 'fwd_packet_length_min', 'fwd_packet_length_mean', 'fwd_packet_length_std', 'bwd_packet_length_max', 'bwd_packet_length_min', 'bwd_packet_length_mean', 'bwd_packet_length_std', 'flow_bytess', 'flow_packetss', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max', 'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std', 'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length', 'bwd_header_length', 'fwd_packetss', 'bwd_packetss', 'packet_length_min', 'packet_length_max', 'packet_length_mean', 'packet_length_std', 'packet_length_variance', 'fin_flag_count', 'syn_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count', 'urg_flag_count', 'cwe_flag_count', 'ece_flag_count', 'downup_ratio',

,count,mean,std,min,25%,50%,75%,max
flow_duration,223112.0,1.643322e+07,3.166018e+07,0.0,81637.75,1.536448e+06,8.957830e+06,1.199999e+08
total_fwd_packets,223112.0,4.905375e+00,1.550975e+01,1.0,2.00,3.000000e+00,5.000000e+00,1.932000e+03
total_backward_packets,223112.0,4.611554e+00,2.188016e+01,0.0,1.00,4.000000e+00,5.000000e+00,2.942000e+03
fwd_packets_length_total,223112.0,9.496634e+02,3.267081e+03,0.0,26.00,3.000000e+01,6.200000e+01,1.830120e+05
bwd_packets_length_total,223112.0,6.029361e+03,3.944391e+04,0.0,0.00,1.640000e+02,1.160100e+04,5.172346e+06
fwd_packet_length_max,223112.0,5.445760e+02,1.874258e+03,0.0,6.00,2.000000e+01,3.500000e+01,1.168000e+04
fwd_packet_length_min,223112.0,2.790173e+01,1.642428e+02,0.0,0.00,0.000000e+00,6.000000e+00,1.472000e+03
fwd_packet_length_mean,223112.0,1.664596e+02,5.076248e+02,0.0,6.00,8.666667e+00,3.200000e+01,3.867000e+03
fwd_packet_length_std,223112.0,2.174395e+02,8.017597e+02,0.0,0.00,5.301991e+00,1.026320e+01,6.692645e+03
bwd_packet_length_max,223112.0,2.767132e+03,3.715450e+03,0.0,0.00,1.000000e+02,5.840000e+03,1.168000e+04



✔ Exploration complete.




In [ ]:
display(cic_proc.head())
display(cicids_proc.head())

,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,...,fwd_act_data_packets,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,48,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,1,2,0,2944.0,0.0,1472.0,1472.0,1472.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,1,2,0,2896.0,0.0,1448.0,1448.0,1448.0,0.0,0.0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,bwd_packet_length_max,...,fwd_act_data_packets,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,3,2,0,12,0,6,6,6.0,0.0,0,...,1,0.0,0.0,0,0,0.0,0.0,0,0,0
1,109,1,1,6,6,6,6,6.0,0.0,6,...,0,0.0,0.0,0,0,0.0,0.0,0,0,0
2,52,1,1,6,6,6,6,6.0,0.0,6,...,0,0.0,0.0,0,0,0.0,0.0,0,0,0
3,34,1,1,6,6,6,6,6.0,0.0,6,...,0,0.0,0.0,0,0,0.0,0.0,0,0,0
4,3,2,0,12,0,6,6,6.0,0.0,0,...,1,0.0,0.0,0,0,0.0,0.0,0,0,0


# 📊 Data Preprocessing Visualization Report

This section generates key visualizations to include in the data preprocessing report.
We focus on:
1. **Missing Values Heatmap**: To show data quality.
2. **Label Distribution**: To show class imbalance.
3. **Correlation Matrix**: To show feature redundancy.
4. **Feature Distributions**: Boxplots for outliers.

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

def plot_missing_heatmap(df, title="Missing Values Heatmap"):
    plt.figure(figsize=(12, 6))
    sns.heatmap(df.isnull(), cbar=False, cmap="viridis", yticklabels=False)
    plt.title(title)
    plt.show()

def plot_label_distribution(df, label_col='label', title="Class Distribution"):
    plt.figure(figsize=(10, 5))
    count = df[label_col].value_counts()
    sns.barplot(x=count.index, y=count.values, palette="viridis")
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.show()

def plot_correlation_matrix(df, title="Feature Correlation Matrix"):
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=['float64', 'int64'])
    # Calculate correlation
    corr = numeric_df.corr()
    # Plot
    plt.figure(figsize=(16, 12))
    sns.heatmap(corr, cmap="coolwarm", annot=False, fmt=".2f")
    plt.title(title)
    plt.show()

# Example usage (assuming 'cic' or 'cic_proc' dataframe exists from previous cells)
if 'cic' in locals():
    print("Plotting for CIC Dataset:")
    # 1. Missing Map (Sampled if too large)
    plot_missing_heatmap(cic.sample(min(10000, len(cic))), title="CIC2019 Missing Values (Sampled)")
    
    # 2. Label Dist
    if 'label' in cic.columns:
        plot_label_distribution(cic, label_col='label', title="CIC2019 Label Distribution")
    elif 'Label' in cic.columns:
        plot_label_distribution(cic, label_col='Label', title="CIC2019 Label Distribution")
        
    # 3. Correlation (Sampled)
    plot_correlation_matrix(cic.sample(min(5000, len(cic))), title="CIC2019 Feature Correlation (Sampled)")
